In [20]:
from leagueScripts import PlayTypeLeagueAverage
from nba_api.stats.library.parameters import PlayType, PlayerOrTeamAbbreviation, TypeGroupingNullable
from nba_api.stats.endpoints import SynergyPlayTypes
import pandas as pd

type_grouping_nullable = TypeGroupingNullable.offensive
player_or_team_abbreviation = PlayerOrTeamAbbreviation.player

playtypes_df = []
for playtype in PlayTypeLeagueAverage.get_playtype_classes_names():
    _playtype_df = SynergyPlayTypes(
        play_type_nullable=getattr(PlayType, playtype), 
        type_grouping_nullable=type_grouping_nullable, player_or_team_abbreviation=player_or_team_abbreviation
    ).synergy_play_type.get_data_frame()
    if not _playtype_df.empty:
        playtypes_df.append(_playtype_df)
df = pd.concat(playtypes_df)
categories = ['TEAM_NAME', 'PLAY_TYPE', 'POSS', 'PPP', 'POSS_PCT']
if player_or_team_abbreviation == PlayerOrTeamAbbreviation.player:
    categories.insert(0, 'PLAYER_NAME') 
df = df[categories]

# Define a custom aggregation function
def custom_agg(group):
    # Calculate the sum of POSS
    poss_sum = group['POSS'].sum()
    
    # Calculate the correct PPP and POSS_PCT
    ppp_numerator = (group['PPP'] * group['POSS']).sum()
    ppp_denominator = poss_sum
    ppp = ppp_numerator / ppp_denominator if ppp_numerator != 0 else 0
    
    poss_pct_numerator = poss_sum
    poss_pct_denominator = (group['POSS'] / group['POSS_PCT']).sum()
    poss_pct = poss_pct_numerator / poss_pct_denominator if poss_pct_denominator != 0 else 0
    
    # Return as a Series
    return pd.Series(
        ({'TEAM_NAME': ', '.join(group['TEAM_NAME'].unique())} if player_or_team_abbreviation == PlayerOrTeamAbbreviation.player else {})
        |
        {
        'POSS': poss_sum,
        'PPP': ppp,
        'POSS_PCT': poss_pct,
        }
    )

# Apply the custom aggregation for players with multiple teams
df = df.groupby(['PLAYER_NAME' if player_or_team_abbreviation == PlayerOrTeamAbbreviation.player else 'TEAM_NAME' , 'PLAY_TYPE']).apply(custom_agg, include_groups=False)
# Calculate percentiles
df['PERCENTILE_FREQ%'] = df.groupby('PLAY_TYPE')['POSS_PCT'].rank(pct=True) * 100

df

TEAM_NAME  POSS    PPP  POSS_PCT  \
PLAYER_NAME     PLAY_TYPE                                                    
AJ Green        Handoff             Milwaukee Bucks    23  1.435     0.110   
                OffScreen           Milwaukee Bucks    20  0.950     0.095   
                PRBallHandler       Milwaukee Bucks    14  0.643     0.067   
                PRRollMan           Milwaukee Bucks    14  1.214     0.067   
                Spotup              Milwaukee Bucks    90  1.233     0.429   
...                                             ...   ...    ...       ...   
Zion Williamson PRBallHandler  New Orleans Pelicans   264  0.943     0.178   
                PRRollMan      New Orleans Pelicans    24  0.917     0.016   
                Postup         New Orleans Pelicans   188  0.894     0.127   
                Spotup         New Orleans Pelicans    91  0.978     0.061   
                Transition     New Orleans Pelicans   204  1.382     0.138   

                               PERCENTILE_FREQ%  
PLAYER_NAME     PLAY_TYPE                        
AJ Green        Handoff               90.449438  
                OffScreen             86.197917  
                PRBallHandler         20.848057  
                PRRollMan             52.500000  
                Spotup                83.569405  
...                                         ...  
Zion Williamson PRBallHandler         55.123675  
                PRRollMan             10.625000  
                Postup                83.333333  
                Spotup                 4.532578  
                Transition            24.852941  

[2929 rows x 5 columns]

In [34]:
# TODO - Check why not 100
df.loc['Aaron Gordon']['POSS_PCT'].sum()

0.8600000000000001

In [21]:
# Pivot the dataframe
df_pivot = df.pivot_table(index=['PLAYER_NAME'], 
                          columns='PLAY_TYPE', 
                          values=['PPP', 'POSS_PCT', 'PERCENTILE_FREQ%'], 
                          aggfunc='first', 
                          fill_value=0)

# Flatten the columns
# df_pivot.columns = [f'{i}_{j}' for i, j in df_pivot.columns]
df_pivot = df_pivot.rename(columns={'PLAYER_NAME_': 'PLAYER_NAME', 'TEAM_NAME_': 'TEAM_NAME'})

df_ppp_for_plot = df_pivot['PPP']
df_frequency_for_plot = df_pivot['POSS_PCT']
df_percentile_frequency_for_plot = df_pivot['PERCENTILE_FREQ%']
metrics = df_ppp_for_plot.columns.tolist()

In [22]:
df_ppp_for_plot

PLAY_TYPE,Cut,Handoff,Isolation,Misc,OffRebound,OffScreen,PRBallHandler,PRRollMan,Postup,Spotup,Transition
PLAYER_NAME,,,,,,,,,,,
AJ Green,0.000,1.435,0.000,0.000,0.000,0.950,0.643,1.214,0.000,1.233,0.000
Aaron Gordon,1.530,0.800,0.638,0.477,0.000,0.000,0.609,1.339,1.037,1.071,1.070
Aaron Holiday,0.000,0.826,1.030,0.000,0.846,0.895,1.027,0.000,0.000,1.131,1.014
Aaron Nesmith,1.138,1.184,0.000,0.500,1.286,1.250,0.000,1.480,0.000,1.162,1.275
Aaron Wiggins,1.302,1.063,0.000,0.455,1.037,0.000,1.056,1.480,0.000,1.260,1.139
...,...,...,...,...,...,...,...,...,...,...,...
Zach Collins,1.356,0.000,1.045,0.253,1.000,0.000,0.000,0.992,0.976,0.885,1.049
Zach LaVine,0.000,1.080,0.833,0.846,0.000,0.875,0.942,0.000,0.000,1.038,1.202
Zavier Simpson,0.000,0.000,0.000,0.000,0.000,0.000,0.667,0.000,0.000,0.000,0.818


In [23]:
df_frequency_for_plot

PLAY_TYPE,Cut,Handoff,Isolation,Misc,OffRebound,OffScreen,PRBallHandler,PRRollMan,Postup,Spotup,Transition
PLAYER_NAME,,,,,,,,,,,
AJ Green,0.000,0.110,0.000,0.000,0.000,0.095,0.067,0.067,0.000,0.429,0.000
Aaron Gordon,0.179,0.021,0.085,0.047,0.000,0.000,0.049,0.060,0.115,0.105,0.199
Aaron Holiday,0.000,0.047,0.067,0.000,0.026,0.038,0.296,0.000,0.000,0.324,0.140
Aaron Nesmith,0.039,0.051,0.000,0.054,0.038,0.059,0.000,0.034,0.000,0.399,0.259
Aaron Wiggins,0.136,0.034,0.000,0.047,0.058,0.000,0.078,0.054,0.000,0.315,0.248
...,...,...,...,...,...,...,...,...,...,...,...
Zach Collins,0.110,0.000,0.027,0.106,0.083,0.000,0.000,0.303,0.150,0.127,0.074
Zach LaVine,0.000,0.053,0.127,0.028,0.000,0.034,0.327,0.000,0.000,0.168,0.221
Zavier Simpson,0.000,0.000,0.000,0.000,0.000,0.000,0.409,0.000,0.000,0.000,0.167


In [24]:
df_percentile_frequency_for_plot

PLAY_TYPE,Cut,Handoff,Isolation,Misc,OffRebound,OffScreen,PRBallHandler,PRRollMan,Postup,Spotup,Transition
PLAYER_NAME,,,,,,,,,,,
AJ Green,0.000000,90.449438,0.000000,0.000000,0.000000,86.197917,20.848057,52.500000,0.000000,83.569405,0.000000
Aaron Gordon,83.274021,8.426966,70.512821,38.424437,0.000000,0.000000,13.780919,49.791667,79.166667,10.623229,65.000000
Aaron Holiday,0.000000,37.265918,57.264957,0.000000,21.654930,48.437500,76.855124,0.000000,0.000000,57.648725,25.441176
Aaron Nesmith,24.911032,44.194757,0.000000,50.803859,37.323944,68.229167,0.000000,33.333333,0.000000,75.920680,92.352941
Aaron Wiggins,77.580071,20.411985,0.000000,38.424437,52.640845,0.000000,23.851590,46.041667,0.000000,54.107649,90.735294
...,...,...,...,...,...,...,...,...,...,...,...
Zach Collins,73.487544,0.000000,15.811966,92.443730,65.492958,0.000000,0.000000,98.750000,90.972222,13.881020,4.705882
Zach LaVine,0.000000,48.127341,85.897436,5.144695,0.000000,41.666667,83.568905,0.000000,0.000000,18.838527,77.352941
Zavier Simpson,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,94.699647,0.000000,0.000000,0.000000,43.823529


In [25]:
from leagueScripts import NBALeague

league_object = NBALeague.get_cached_league_object()

top_scorers = sorted(league_object.players_on_teams_objects_list, key=lambda x: x.stats_df['PTS'].sum().item() if not x.stats_df.empty else 0, reverse=True)[:100]
players_to_plot = [player.player_info['DISPLAY_FIRST_LAST'].item() for player in top_scorers]
title = f"{', '.join(players_to_plot) if len(players_to_plot) <5 else 'Players'} play type stats"

In [26]:
import numpy as np
import plotly.graph_objects as go

# Create a new Figure
fig = go.Figure()

# Add averages
fig.add_trace(go.Scatterpolar(
        r=df_ppp_for_plot.replace(0, np.NaN).mean(),
        theta=metrics,
        name="Average",
        line=dict(color='white'),  # Set the color for the average trace
        showlegend=False  # Make the Average trace always visible and not part of the legend
    ))

# Plot each player in a separate chart
for player_name in players_to_plot:
    player_ppp_df = df_ppp_for_plot.loc[player_name]
    player_frequency_df = df_frequency_for_plot.loc[player_name]
    player_percentile_frequency_df = df_percentile_frequency_for_plot.loc[player_name]

    fig.add_trace(go.Scatterpolar(
        r=player_ppp_df.values,
        theta=metrics,
        fill='toself',
        name=player_name,
        marker=dict(
            size=player_percentile_frequency_df.values
        ),
        customdata=player_frequency_df.values * 100,  # Adding custom data
        hovertemplate='<b>%{theta}</b><br>PPP: %{r}<br>Frequency: %{customdata}%<extra></extra>'
    
    ))

# Update layout with increased width and height
fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 2]
        )),
    showlegend=True,
    title=title,
    width=1500,
    height=1000,
)

# Show the plot
fig.show()
